# Setup

In [ ]:
%%bash
/home/airflow/.venv/bin/python ./generate_data.py
/home/airflow/.venv/bin/python ./run_ddl.py

In [2]:
%%sql --show
use prod_db

""


In [3]:
%%sql
SELECT
    l_orderkey,
    round( sum(l_extendedprice * (1 - l_discount) * (1 + l_tax)),
        2
    ) AS totalprice
FROM
    lineitem
WHERE
    l_orderkey = 1
GROUP BY
    l_orderkey;


,l_orderkey,totalprice
0,1,194029.59


In [4]:
%%sql

-- The totalprice of an order (with orderkey = 1)
SELECT
    o_orderkey,
    o_totalprice
FROM
    orders
WHERE
    o_orderkey = 1;



,o_orderkey,o_totalprice
0,1,194029.55


In [5]:
%%sql
    CREATE TABLE dim_customer AS 
 SELECT 
        c.*,
        n_name AS nation_name,
        n_comment AS nation_comment,
        r_name AS region_name,
        r_comment AS region_comment
    FROM customer c
    LEFT JOIN nation n ON c_nationkey = n_nationkey
    LEFT JOIN region r ON n_regionkey = r_regionkey

""


In [6]:
%%sql
CREATE TABLE fct_orders as select * from orders

""


In [7]:
%%sql
CREATE TABLE fct_lineitem as select * from lineitem

""


In [8]:
%%sql
create table wide_orders as 
 SELECT 
        f.*,
        d.*
    FROM fct_orders f
    LEFT JOIN dim_customer d ON f.o_custkey = d.c_custkey

""


In [9]:
%%sql
create table wide_lineitem as select * from fct_lineitem

""


In [10]:
%%sql
create table order_lineitem_metrics as 
SELECT 
        l_orderkey as order_key,
        COUNT(l_linenumber) AS num_lineitems
    FROM wide_lineitem
    GROUP BY l_orderkey

""


In [11]:
%%sql
create table customer_outreach_metrics as 
SELECT 
        wo.c_custkey,
        wo.c_name AS customer_name,
        MIN(wo.o_totalprice) AS min_order_value,
        MAX(wo.o_totalprice) AS max_order_value,
        AVG(wo.o_totalprice) AS avg_order_value,
        AVG(olm.num_lineitems) AS avg_num_items_per_order
    FROM wide_orders wo
    LEFT JOIN order_lineitem_metrics olm ON wo.o_orderkey = olm.order_key
    GROUP BY wo.c_custkey, wo.c_name

""


In [12]:
%%sql --show
select *
from customer_outreach_metrics

,c_custkey,customer_name,min_order_value,max_order_value,avg_order_value,avg_num_items_per_order
0,13678,Customer#000013678,3643.59,447542.26,150829.488571,3.666667
1,12484,Customer#000012484,3871.10,261383.06,122749.683636,3.545455
2,7940,Customer#000007940,53746.57,320282.96,165889.345455,4.909091
3,4945,Customer#000004945,2205.73,276173.48,127629.438947,3.842105
4,13276,Customer#000013276,58779.29,300437.66,149012.655238,4.000000
...,...,...,...,...,...,...
995,11326,Customer#000011326,54734.25,337839.46,171229.530000,5.000000
996,10553,Customer#000010553,41593.74,378066.38,146551.840000,3.714286
997,6380,Customer#000006380,57676.61,188571.97,117654.280000,3.000000
998,1265,Customer#000001265,38886.52,299235.43,184031.156250,4.625000
